# Qwen2.5-VL-7B — V-STaR + Region Caption Inference

Sanity-check notebook: load a V-STaR sample via `VStarDataset`, subsample 32 frames, preview them as an inline video, then send the frames + question + region caption to the local vLLM server.

**Prerequisites:** launch the server with `bash bash_scripts/vllm_serving/trained_ckpts.sh` (set `MODEL=Qwen/Qwen2.5-VL-7B-Instruct` and clear `LORA_MODULE` for the base model, or point it at a LoRA checkpoint).

## Config

In [115]:
# --- paths ---
CONFIG_PATH        = "/projects/torresani-lab/ajay/visual-grounded-vlm/config/infer/qwen25vl7b_vstar_rcap_vg/default.yaml"
LORA_EXPERIMENT_DIR = ""   # set to LoRA checkpoint dir, or leave empty for base model

# Problematic question framing = 2, 6
# RCAP biases reasoner = 8

# --- sample selection ---
SAMPLE_IDX = 14

# --- server ---
OPENAI_API_BASE = "http://localhost:8000/v1"
# Use "sft" when serving with LoRA (matches LORA_NAME in trained_ckpts.sh).
# Use "Qwen/Qwen2.5-VL-7B-Instruct" when serving without LoRA.
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

## 1. Load config and dataset

In [116]:
import sys
sys.path.append("/projects/torresani-lab/ajay/visual-grounded-vlm/")

import os
import json
import yaml

import numpy as np
import torch

from src.dataset.vstar import VStarDataset

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# resolve model path (from LoRA adapter config if provided)
if LORA_EXPERIMENT_DIR:
    with open(os.path.join(LORA_EXPERIMENT_DIR, "adapter_config.json")) as f:
        adapter_cfg = json.load(f)
    model_path = adapter_cfg["base_model_name_or_path"]
else:
    model_path = cfg["model"]["path"]

RCAP_CAPTIONS_ROOT = cfg["rcap_captions_root"]
TGT_FRAMES = cfg["tgt_frames"]

ZEROSHOT_RESULTS_PATH = "/scratch/rajendrakumar.a/visual-grounded-vlm/results/qwen25vl7bi_vstar_zeroshot/2026-05-18_18-35-15/results.json"

# load zeroshot results and index by qid
with open(ZEROSHOT_RESULTS_PATH) as f:
    zeroshot_results = {r["qid"]: r["generated_answer"] for r in json.load(f)}

dataset = VStarDataset(**cfg["dataset"])
print(f"Loaded {len(dataset)} V-STaR samples")
print(f"Model: {model_path}")
print(f"RCAP root: {RCAP_CAPTIONS_ROOT}")
print(f"Target frames: {TGT_FRAMES}")

Loaded 2094 V-STaR samples
Model: Qwen/Qwen2.5-VL-7B-Instruct
RCAP root: /scratch/rajendrakumar.a/visual-grounded-vlm/annotations/vstar/vstar_qwen3vl32b_rcap
Target frames: 32


## 2. Pick a sample, subsample frames, and load the region caption

In [117]:
sample = dataset[SAMPLE_IDX]
print(f"qid: {sample['qid']}  vid: {sample['vid']}")
print(f"Question: {sample['question']}")
print(f"GT answer: {sample.get('answer', sample.get('gt_answer'))}")

# uniformly subsample to TGT_FRAMES (mirrors qwen_rcap_infer_vg.py)
frames = sample["frames"]  # (N, C, H, W) uint8 tensor
if frames.shape[0] > TGT_FRAMES:
    indices = np.round(np.linspace(0, frames.shape[0] - 1, TGT_FRAMES)).astype(int)
    frames = frames[indices]
print(f"Frames after subsampling: {frames.shape}")

# load region caption
rcap_path = os.path.join(RCAP_CAPTIONS_ROOT, "samples", f"{sample['qid']}.json")
with open(rcap_path) as f:
    rcap = json.load(f)
print(f"\nRegion caption:\n{rcap['rcap']}")

qid: 4013751996_0  vid: 4013751996
Question: who rides the blue toy?
GT answer: a child in pink rides a blue toy.
Frames after subsampling: torch.Size([30, 3, 280, 392])

Region caption:
<caption>A young child wearing a bright green dinosaur-themed helmet and a pink long-sleeve shirt is riding a blue toy vehicle, likely a tricycle or ride-on toy, along a paved path in a backyard. The child moves forward, then stops and turns slightly toward the camera, eventually looking directly at it with a calm expression. The red bounding box consistently tracks the child throughout the sequence, capturing their upper body and head as they interact with the environment, which includes a wooden fence, grass, and garden elements. The blue toy is visible beneath the child during the initial frames but becomes less prominent as the focus shifts to the child’s face and helmet in later frames.</caption>


## 3. Preview the sampled frames as an inline video

In [118]:
import io
import base64
from IPython.display import HTML, display
from torchcodec.encoders import VideoEncoder

# encode sampled frames to a temporary mp4 in memory
dump_dir = "/projects/torresani-lab/ajay/visual-grounded-vlm/scripts/vllm_serving/dump"
os.makedirs(dump_dir, exist_ok=True)
preview_path = os.path.join(dump_dir, f"preview_{sample['qid']}.mp4")
VideoEncoder(frames.to(torch.uint8), frame_rate=4).to_file(preview_path)

# embed as base64 so it plays inline without a file server
with open(preview_path, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<p><b>qid:</b> {sample['qid']} &nbsp;|&nbsp; <b>frames:</b> {frames.shape[0]} &nbsp;|&nbsp; <b>size:</b> {frames.shape[3]}×{frames.shape[2]}</p>
<video controls loop width="640">
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
<p><b>Question:</b> {sample['question']}</p>
<p><b>Region caption:</b> {rcap['rcap']}</p>
"""))

## 4. Build the chat request

In [119]:
from openai import OpenAI

client = OpenAI(api_key="EMPTY", base_url=OPENAI_API_BASE)

question = sample['question']
# question = " who is on the watercraft on the water?"

caption = rcap['rcap']
user_text = f"Question: {question}\nRegion Caption: {caption}"
with open("/projects/torresani-lab/ajay/visual-grounded-vlm/prompts/inference/bbox_rcap_prompt_v3.md") as f:
    SYSTEM_PROMPT = f.read()

message = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": user_text},
            {
                "type": "video_url",
                "video_url": {"url": "file://" + preview_path},
                "VIDEO_MAX_PIXELS": cfg["dataset"]["video_config"]["max_pixels"],
                "fps": cfg["dataset"]["video_config"]["fps"],
            },
        ],
    },
]

print("\nUser prompt:")
print(user_text)


User prompt:
Question: who rides the blue toy?
Region Caption: <caption>A young child wearing a bright green dinosaur-themed helmet and a pink long-sleeve shirt is riding a blue toy vehicle, likely a tricycle or ride-on toy, along a paved path in a backyard. The child moves forward, then stops and turns slightly toward the camera, eventually looking directly at it with a calm expression. The red bounding box consistently tracks the child throughout the sequence, capturing their upper body and head as they interact with the environment, which includes a wooden fence, grass, and garden elements. The blue toy is visible beneath the child during the initial frames but becomes less prominent as the focus shifts to the child’s face and helmet in later frames.</caption>


## 5. Send request and print response

In [120]:
chat_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=message,
    max_tokens=cfg["sampling"]["max_tokens"],
    temperature=cfg["sampling"]["temperature"],
    seed=cfg["sampling"]["seed"],
)
response_text = chat_response.choices[0].message.content
print(response_text)

<think>
The region description clearly states that a young child is riding a blue toy vehicle, likely a tricycle or ride-on toy, and the video frames confirm this. The child's movements and the presence of the toy under them align with the description.
</think>
<answer>
A young child is riding the blue toy. </answer>


In [121]:
print(f"Question:       {sample['question']}")
print(f"GT answer:      {sample.get('answer', sample.get('gt_answer'))}")
print(f"Region caption: {rcap['rcap']}")

zeroshot_answer = zeroshot_results.get(sample["qid"], "(not found)")
print(f"\nZeroshot answer: {zeroshot_answer}")

Question:       who rides the blue toy?
GT answer:      a child in pink rides a blue toy.
Region caption: <caption>A young child wearing a bright green dinosaur-themed helmet and a pink long-sleeve shirt is riding a blue toy vehicle, likely a tricycle or ride-on toy, along a paved path in a backyard. The child moves forward, then stops and turns slightly toward the camera, eventually looking directly at it with a calm expression. The red bounding box consistently tracks the child throughout the sequence, capturing their upper body and head as they interact with the environment, which includes a wooden fence, grass, and garden elements. The blue toy is visible beneath the child during the initial frames but becomes less prominent as the focus shifts to the child’s face and helmet in later frames.</caption>

Zeroshot answer: A child rides the blue toy.
